In [1]:
import os
import json
import ast
import subprocess
import sys
import py_compile
from pathlib import Path
from typing import TypedDict, Optional, Dict, Any, List

from getpass import getpass
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

# Load environment variables
load_dotenv()

# Force UTF-8 encoding on Windows standard streams to prevent UnicodeEncodeErrors when printing emojis
if sys.platform.startswith('win'):
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8')
    if hasattr(sys.stderr, 'reconfigure'):
        sys.stderr.reconfigure(encoding='utf-8')

# Global configuration
AGENT_FILE = "generated_agent.py"

c:\Users\hp\OneDrive\Desktop\utsarjan\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Fast model → requirements, tool selection
fast_model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Strong model → architecture, specification
reasoning_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Coding model → code generation
coding_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

print("All models initialized successfully.")

All models initialized successfully.


In [3]:
class AgentState(TypedDict):
    user_prompt: str
    requirements: str
    architecture: str
    tools: str
    agent_spec: Dict[str, Any]
    generated_code: Optional[str]
    validation_result: Optional[Dict[str, Any]]

In [4]:
TOOL_REGISTRY = {
    "pdf_loader": {
        "package": "PyMuPDF",
        "import": "import pymupdf",
        "implementation": "pymupdf.open",
        "purpose": "Load PDF documents."
    },

    "pdf_text_extractor": {
        "package": "PyMuPDF",
        "import": "import pymupdf",
        "implementation": "page.get_text()",
        "purpose": "Extract text from PDF pages."
    },

    "ocr": {
        "package": "pytesseract",
        "import": "import pytesseract",
        "implementation": "pytesseract.image_to_string",
        "purpose": "Extract text from scanned PDF pages."
    },

    "document_chunker": {
        "package": "langchain-text-splitters",
        "import": "from langchain_text_splitters import RecursiveCharacterTextSplitter",
        "implementation": "RecursiveCharacterTextSplitter",
        "purpose": "Split documents into chunks."
    },

    "embedding_generator": {
        "package": "sentence-transformers",
        "import": "from sentence_transformers import SentenceTransformer",
        "implementation": "SentenceTransformer",
        "purpose": "Generate embeddings."
    },

    "vector_store": {
        "package": "faiss-cpu",
        "import": "import faiss",
        "implementation": "faiss.IndexFlatL2",
        "purpose": "Store and search embeddings."
    },

    "retriever": {
        "package": "faiss-cpu",
        "import": "import faiss",
        "implementation": "index.search",
        "purpose": "Retrieve relevant document chunks."
    },

    "citation_generator": {
        "package": "Python",
        "import": "",
        "implementation": "custom function",
        "purpose": "Generate page citations."
    },

    "llm": {
        "package": "langchain-groq",
        "import": "from langchain_groq import ChatGroq",
        "implementation": "ChatGroq(model=\"openai/gpt-oss-120b\", temperature=0)",
        "purpose": "Generate answers using Groq."
    }
}

print("Available tools:")
for name in TOOL_REGISTRY:
    print("-", name)

Available tools:
- pdf_loader
- pdf_text_extractor
- ocr
- document_chunker
- embedding_generator
- vector_store
- retriever
- citation_generator
- llm


In [5]:
def requirement_agent(state: AgentState):
    prompt = f"""
You are a Requirement Analysis Agent.

Analyze this user request:

{state["user_prompt"]}

Return:

1. Agent Goal
2. Inputs
3. Outputs
4. Required Capabilities
5. Required Tools
6. Memory Requirement
7. Constraints
8. Success Criteria

Keep the response structured and concise.
Do not generate code.
"""
    response = fast_model.invoke(prompt)
    return {
        "requirements": response.content
    }

In [6]:
def architecture_agent(state: AgentState):
    tools = "\n".join(
        f"- {name}: {info['purpose']}"
        for name, info in TOOL_REGISTRY.items()
    )

    prompt = f"""
You are a Senior AI Agent Architect.

User Request:
{state["user_prompt"]}

Requirements:
{state["requirements"]}

Available Tools:
{tools}

Design an implementable architecture using Python
and LangGraph.

Include:

1. Architecture pattern
2. Nodes
3. Node responsibilities
4. Workflow
5. State variables
6. Tools
7. Error handling
8. Validation

Only use tools from the provided list.

Do NOT generate code.
"""
    response = reasoning_model.invoke(prompt)
    return {
        "architecture": response.content
    }

In [7]:
def tool_selection_agent(state: AgentState):
    tools = "\n".join(
        f"- {name}: {info['purpose']}"
        for name, info in TOOL_REGISTRY.items()
    )

    prompt = f"""
You are a Tool Selection Agent.

User Requirements:
{state["requirements"]}

Architecture:
{state["architecture"]}

Available Tools:
{tools}

Select ONLY the tools actually required.

Return only this format:

REQUIRED TOOLS:
tool_name
tool_name
tool_name

Do not invent tools.
"""
    response = fast_model.invoke(prompt)
    return {
        "tools": response.content
    }

In [8]:
def agent_spec_generator(state: AgentState):
    prompt = f"""
You are a Senior AI Agent Specification Designer.

Create a strict JSON specification.

USER PROMPT:
{state["user_prompt"]}

REQUIREMENTS:
{state["requirements"]}

ARCHITECTURE:
{state["architecture"]}

SELECTED TOOLS:
{state["tools"]}

Allowed model:

Provider: groq
Model:
openai/gpt-oss-20b

or

openai/gpt-oss-120b

Return ONLY valid JSON.

Structure:

{{
    "agent_name": "",
    "description": "",
    "goal": "",
    "input": {{}},
    "output": {{}},
    "model": {{
        "provider": "groq",
        "model": "",
        "temperature": 0
    }},
    "tools": [],
    "workflow": [],
    "error_handling": [],
    "success_criteria": []
}}

Rules:

- Never use OpenAI models.
- Never use Gemini.
- Never invent tools.
- Tools must come from the provided registry.
"""
    response = reasoning_model.invoke(prompt)
    content = response.content.strip()
    if content.startswith("```"):
        content = (
            content
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )
    return {
        "agent_spec": json.loads(content)
    }

In [9]:
APPROVED_IMPORTS = {
    "pymupdf": None,
    "pytesseract": None,
    "faiss": None,
    "numpy": None,
    "PIL": {"Image"},
    "io": None,
    "langchain_groq": {
        "ChatGroq"
    },
    "langchain_text_splitters": {
        "RecursiveCharacterTextSplitter"
    },
    "sentence_transformers": {
        "SentenceTransformer"
    },
    "langgraph.graph": {
        "StateGraph",
        "START",
        "END"
    },
    "os": None,
    "json": None,
    "typing": None,
    "pathlib": None
}

def validate_imports(code):
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return False, f"Syntax error: {e}"

    errors = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name not in APPROVED_IMPORTS:
                    errors.append(
                        f"Unapproved import: {alias.name}"
                    )
        elif isinstance(node, ast.ImportFrom):
            module = node.module or ""
            if module not in APPROVED_IMPORTS:
                errors.append(
                    f"Unapproved module: {module}"
                )
                continue
            allowed_names = APPROVED_IMPORTS[module]
            if allowed_names is not None:
                for alias in node.names:
                    if alias.name not in allowed_names:
                        errors.append(
                            f"Unapproved import: "
                            f"{alias.name} from {module}"
                        )
    if errors:
        return False, "\n".join(errors)
    return True, "Imports are valid."

In [10]:
def validate_syntax(code):
    try:
        ast.parse(code)
        return True, "Syntax is valid."
    except SyntaxError as e:
        return False, (
            f"{e.msg} "
            f"at line {e.lineno}, column {e.offset}"
        )

In [11]:
def validate_tool_implementation(code, agent_spec):
    selected_tools = agent_spec.get("tools", [])
    errors = []

    if "pdf_loader" in selected_tools:
        if "pymupdf.open" not in code:
            errors.append("pdf_loader selected but pymupdf.open() was not found")

    if "pdf_text_extractor" in selected_tools:
        if "get_text" not in code:
            errors.append("pdf_text_extractor selected but page.get_text() was not found")

    if "ocr" in selected_tools:
        if "image_to_string" not in code:
            errors.append("ocr selected but pytesseract.image_to_string() was not found")

    if "document_chunker" in selected_tools:
        if "RecursiveCharacterTextSplitter" not in code:
            errors.append("document_chunker selected but RecursiveCharacterTextSplitter was not found")

    if "embedding_generator" in selected_tools:
        if "SentenceTransformer" not in code:
            errors.append("embedding_generator selected but SentenceTransformer was not found")

    if "vector_store" in selected_tools:
        if "IndexFlatL2" not in code and "faiss.Index" not in code:
            errors.append("vector_store selected but faiss.IndexFlatL2 was not found")

    if "retriever" in selected_tools:
        if ".search(" not in code and "index.search" not in code:
            errors.append("retriever selected but faiss index.search() was not found")

    if "citation_generator" in selected_tools:
        if "citation" not in code.lower() and "source" not in code.lower():
            errors.append("citation_generator selected but citation/metadata handling logic was not found")

    if "llm" in selected_tools:
        if "ChatGroq" not in code:
            errors.append("llm selected but ChatGroq was not found")

    if errors:
        return False, "\n".join(errors)
    return True, "All selected tools are implemented."

In [12]:
def validate_requirements(code, agent_spec, user_prompt):
    prompt = f"""
You are a Software QA Agent.
Verify if the following generated Python code satisfies the functional requirements of the user prompt and the selected tools in agent specification.

USER PROMPT:
{user_prompt}

AGENT SPECIFICATION:
{json.dumps(agent_spec, indent=4)}

CODE:
{code}

Analyze if the code implements the requested pipeline and functional goals.
Specifically:
1. If PDF processing is requested, does it correctly load binary data with pymupdf (not decode("utf-8")), extract text from pages, and preserve page metadata (document_id, file_name, page_number, chunk_id)?
2. If OCR is requested/fallback is needed, does it fall back to pytesseract.image_to_string when page.get_text() is empty?
3. If FAISS vector store is requested, does it use faiss.IndexFlatL2, index.add, and index.search (rather than manual distance loops)?
4. If citations are requested, do they reference actual PDF pages (page.number + 1) and look like "document.pdf, page 7"?

Return your response in this exact JSON format:
{{
    "valid": true/false,
    "errors": [
        "Description of error 1",
        "Description of error 2"
    ]
}}
Do not include any other text, markdown blocks, or explanation. Only return the JSON.
"""
    try:
        response = reasoning_model.invoke(prompt)
        content = response.content.strip()
        if content.startswith("```"):
            content = content.replace("```json", "").replace("```", "").strip()
        result = json.loads(content)

        heuristics_errors = []
        if "pdf" in user_prompt.lower():
            if "pymupdf.open" not in code:
                heuristics_errors.append("PDF agent must use pymupdf.open()")
            if ".decode(" in code and ("pymupdf" not in code or "decode" in code):
                heuristics_errors.append("PDF files should not be decoded as utf-8 strings")

        errors = result.get("errors", [])
        for err in heuristics_errors:
            if err not in errors:
                errors.append(err)

        valid = result.get("valid", True) and not heuristics_errors
        if errors:
            valid = False

        return valid, "\n".join(errors) if errors else "Requirements are satisfied."
    except Exception as e:
        errors = []
        if "pdf" in user_prompt.lower():
            if "pymupdf.open" not in code:
                errors.append("PDF agent must use pymupdf.open()")
        if errors:
            return False, "\n".join(errors)
        return True, "Requirements are satisfied (fallback)."

In [13]:
def validate_runtime(code, agent_spec):
    try:
        Path(AGENT_FILE).write_text(code, encoding="utf-8")

        try:
            py_compile.compile(AGENT_FILE, doraise=True)
        except py_compile.PyCompileError as e:
            return False, f"Compilation failed: {e.msg}"

        proc = subprocess.run(
            [sys.executable, AGENT_FILE],
            capture_output=True,
            text=True,
            timeout=30
        )

        if proc.returncode != 0:
            return False, f"Execution failed with exit code {proc.returncode}.\nStderr:\n{proc.stderr}\nStdout:\n{proc.stdout}"

        stdout = proc.stdout.strip()
        if not stdout:
            return False, "Execution succeeded but stdout was empty. Make sure the __main__ block prints the test output."

        found_json = False
        for line in stdout.splitlines():
            try:
                line_clean = line.strip()
                if line_clean.startswith("{") and line_clean.endswith("}"):
                    data = json.loads(line_clean)
                    if "answer_text" in data or "confidence_score" in data:
                        found_json = True
                        break
            except Exception:
                continue

        if not found_json and ("confidence" not in stdout.lower() and "answer" not in stdout.lower()):
            return False, f"Execution succeeded, but no valid agent output structure was detected in stdout.\nStdout was:\n{stdout}"

        return True, "Runtime validation passed."
    except Exception as e:
        return False, f"Runtime validation exception: {str(e)}"

In [14]:
def code_generator_agent(agent_spec):
    selected_tools = agent_spec.get("tools", [])
    tool_information = []

    for tool in selected_tools:
        if tool in TOOL_REGISTRY:
            info = TOOL_REGISTRY[tool]
            tool_information.append(
                f"""
TOOL: {tool}

Package: {info["package"]}

Approved Import:
{info["import"]}

Approved Implementation:
{info["implementation"]}

Purpose:
{info["purpose"]}
"""
            )

    tools_text = "\n".join(tool_information)

    prompt = f"""
You are a Senior Python AI Agent Developer.

Generate a complete Python implementation from this Agent Specification:

{json.dumps(agent_spec, indent=4)}

APPROVED TOOL IMPLEMENTATIONS:

{tools_text}

STRICT RULES:
1. Return ONLY Python code. No Markdown (do not wrap in ```python or ``` blocks, no backticks).
2. Do not invent imports. Use ONLY the approved imports:
   - import pymupdf (never fitz)
   - import pytesseract
   - import faiss
   - import numpy as np
   - from PIL import Image
   - import io
   - from langchain_groq import ChatGroq
   - from langchain_text_splitters import RecursiveCharacterTextSplitter
   - from sentence_transformers import SentenceTransformer
   - from langgraph.graph import StateGraph, START, END
   - os, json, typing, pathlib
3. Passing syntax and import validation is NOT sufficient. The code must actually implement every selected tool:
   - If "pdf_loader" is selected, the code must open PDF documents from bytes (pymupdf.open(stream=file_bytes, filetype="pdf")) or file paths.
   - If "pdf_text_extractor" is selected, the code must loop through doc pages and use page.get_text().
   - If "ocr" is selected, you must implement an OCR fallback: if page.get_text() returns empty/insufficient text, render the page (pix = page.get_pixmap()) and convert it to PIL Image to run pytesseract.image_to_string(img).
     Example for rendering page to PIL Image:
     pix = page.get_pixmap()
     img_data = pix.tobytes("png")
     img = Image.open(io.BytesIO(img_data))
     text = pytesseract.image_to_string(img)
   - If "document_chunker" is selected, use RecursiveCharacterTextSplitter to split text, preserving chunk metadata: {{"document_id": ..., "file_name": ..., "page_number": page.number + 1, "chunk_id": ...}}.
   - If "embedding_generator" is selected, use SentenceTransformer to embed chunks.
   - If "vector_store" is selected, use faiss.IndexFlatL2 and index.add (never implement manual Euclidean distance loop). Convert embeddings to float32 numpy arrays: np.array(embeddings).astype('float32').
   - If "retriever" is selected, use index.search to retrieve chunks and map them back to metadata.
   - If "citation_generator" is selected, citations must refer to actual PDF page numbers (page.number + 1), formatted like "[filename.pdf, page X]".
4. Always include a test execution block under `if __name__ == "__main__":` that initializes the agent and executes a test turn.
   - If PDF tools are selected, the test block should use pymupdf to create a temporary "test_temp.pdf" (using pymupdf: doc = pymupdf.open(); page = doc.new_page(); page.insert_text((50, 50), "Python is a high-level programming language."); doc.save("test_temp.pdf")) in the local directory, load it as bytes or file path, run the agent with this input, print the JSON output of the turn, and clean up the temporary PDF.
5. Use:
ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
6. Never use deprecated LangChain imports such as:
   - from langchain.chat_models import ChatGroq
   - from langchain.vectorstores import FAISS
   - from langchain.embeddings import HuggingFaceEmbedembeddings
   - from langchain.text_splitter import RecursiveCharacterTextSplitter
   - from langchain.retrievers import SemanticSimilarityRetriever
7. Never hardcode API keys.
"""

    response = coding_model.invoke(prompt)
    code = response.content.strip()

    if code.startswith("```"):
        code = (
            code
            .replace("```python", "")
            .replace("```", "")
            .strip()
        )

    return code

In [15]:
def debugger_agent(code, error):
    prompt = f"""
You are a Python debugging agent.

Fix the following generated code.

ERROR:
{error}

CODE:
{code}

Rules:
1. Return ONLY corrected Python code. No Markdown (do not wrap in ```python or ``` blocks, no backticks).
2. Fix the error while ensuring all selected tools are actually implemented.
3. Do not invent imports. Use ONLY approved imports:
   - import pymupdf (never fitz)
   - import pytesseract
   - import faiss
   - import numpy as np
   - from PIL import Image
   - import io
   - from langchain_groq import ChatGroq
   - from langchain_text_splitters import RecursiveCharacterTextSplitter
   - from sentence_transformers import SentenceTransformer
   - from langgraph.graph import StateGraph, START, END
   - os, json, typing, pathlib
4. Ensure PDF loading uses pymupdf.open(stream=file_bytes, filetype="pdf").
5. Ensure FAISS uses faiss.IndexFlatL2, index.add, and index.search.
6. Ensure citations reference actual PDF page numbers (page.number + 1).
7. Ensure OCR fallback renders the page and uses pytesseract.image_to_string.
8. Maintain a main test block (`if __name__ == "__main__":`) that executes a test turn (creating a temporary test PDF if needed) and prints the result JSON.
9. Do not hardcode API keys.
"""
    response = reasoning_model.invoke(prompt)
    fixed_code = response.content.strip()

    if fixed_code.startswith("```"):
        fixed_code = (
            fixed_code
            .replace("```python", "")
            .replace("```", "")
            .strip()
        )

    return fixed_code

In [16]:
def build_agent(user_prompt):
    state = AgentState(
        user_prompt=user_prompt,
        requirements="",
        architecture="",
        tools="",
        agent_spec={},
        generated_code=None,
        validation_result=None
    )

    state.update(requirement_agent(state))
    print("Requirement Agent       ✅")

    state.update(architecture_agent(state))
    print("Architecture Agent      ✅")

    state.update(tool_selection_agent(state))
    print("Tool Selection          ✅")

    state.update(agent_spec_generator(state))
    print("Agent Specification     ✅")

    code = code_generator_agent(state["agent_spec"])
    state["generated_code"] = code
    print("Code Generation         ✅")

    MAX_RETRIES = 3

    for attempt in range(MAX_RETRIES + 1):
        syntax_ok, syntax_msg = validate_syntax(code)
        if not syntax_ok:
            print("Syntax Validation       ❌")
            if attempt < MAX_RETRIES:
                code = debugger_agent(code, f"Syntax Error: {syntax_msg}")
                continue
            break
        print("Syntax Validation       ✅")

        import_ok, import_msg = validate_imports(code)
        if not import_ok:
            print("Import Validation       ❌")
            if attempt < MAX_RETRIES:
                code = debugger_agent(code, f"Import Error: {import_msg}")
                continue
            break
        print("Import Validation       ✅")
        
        tool_ok, tool_msg = validate_tool_implementation(code, state["agent_spec"])
        if not tool_ok:
            print("Tool Validation         ❌")
            if attempt < MAX_RETRIES:
                code = debugger_agent(code, f"Tool Implementation Error:\n{tool_msg}")
                continue
            break
        print("Tool Validation         ✅")

        req_ok, req_msg = validate_requirements(code, state["agent_spec"], state["user_prompt"])
        if not req_ok:
            print("Requirement Validation  ❌")
            if attempt < MAX_RETRIES:
                code = debugger_agent(code, f"Requirement Error:\n{req_msg}")
                continue
            break
        print("Requirement Validation  ✅")

        runtime_ok, runtime_msg = validate_runtime(code, state["agent_spec"])
        if not runtime_ok:
            print("Runtime Validation      ❌")
            if attempt < MAX_RETRIES:
                code = debugger_agent(code, f"Runtime Error:\n{runtime_msg}")
                continue
            break
        print("Runtime Validation      ✅")

        state["generated_code"] = code
        state["validation_result"] = {
            "valid": True,
            "attempts": attempt + 1
        }
        return state

    state["generated_code"] = code
    state["validation_result"] = {
        "valid": False,
        "attempts": MAX_RETRIES + 1
    }
    return state

In [23]:
result = build_agent(
    "Create an AI agent that reads PDF files and answers "
    "questions from those PDFs with page-level citations."
)

Requirement Agent       ✅
Architecture Agent      ✅
Tool Selection          ✅
Agent Specification     ✅
Code Generation         ✅
Syntax Validation       ✅
Import Validation       ✅
Tool Validation         ✅
Requirement Validation  ❌
Syntax Validation       ✅
Import Validation       ✅
Tool Validation         ✅
Requirement Validation  ❌
Syntax Validation       ✅
Import Validation       ✅
Tool Validation         ✅
Requirement Validation  ✅
Runtime Validation      ❌
Syntax Validation       ✅
Import Validation       ✅
Tool Validation         ✅
Requirement Validation  ✅
Runtime Validation      ❌


In [18]:
print("\n" + "=" * 70)
print("AGENT SPECIFICATION")
print("=" * 70)
print(json.dumps(result["agent_spec"], indent=4))

print("\n" + "=" * 70)
print("GENERATED CODE")
print("=" * 70)
print(result["generated_code"])

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)
print(result["validation_result"])


AGENT SPECIFICATION
{
    "agent_name": "PDFCitationQAAgent",
    "description": "Answers user questions based on the content of one or more PDF documents, providing exact page\u2011level citations and optional excerpts for verification.",
    "goal": "Provide accurate, context\u2011aware answers to user queries using PDF content, citing the exact page(s) from which each answer is derived.",
    "input": {
        "pdf_files": {
            "type": "list",
            "description": "One or more PDF documents supplied as file uploads or URLs."
        },
        "user_query": {
            "type": "string",
            "description": "Natural\u2011language question about the PDF content."
        },
        "optional_metadata": {
            "type": "object",
            "description": "Optional identifiers such as document title, author, or custom tags."
        }
    },
    "output": {
        "answer_text": {
            "type": "string",
            "description": "Concise answer 

In [19]:
agent_code = result["generated_code"]
agent_path = Path(AGENT_FILE)
agent_path.write_text(agent_code, encoding="utf-8")

print(f"Agent saved to: {agent_path}")
print(f"Code size: {len(agent_code)} characters")

Agent saved to: generated_agent.py
Code size: 9030 characters


In [20]:
try:
    py_compile.compile(AGENT_FILE, doraise=True)
    print("✅ Python compilation successful!")
except py_compile.PyCompileError as e:
    print("❌ Compilation failed:")
    print(e)

❌ Compilation failed:
  File "generated_agent.py", line 238
    answer_with_citations = f"{raw}
                            ^
SyntaxError: unterminated string literal (detected at line 238)



In [21]:
import importlib.util

packages = {
    "pymupdf": "PyMuPDF",
    "pytesseract": "pytesseract",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "langchain_groq": "langchain-groq",
    "langchain_text_splitters": "langchain-text-splitters",
    "langgraph": "langgraph"
}

print("\nPACKAGE CHECK")
print("=" * 50)
missing = []
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        print(f"❌ {package}")
        missing.append(package)
    else:
        print(f"✅ {package}")

if missing:
    print("\nMissing packages:")
    for package in missing:
        print("-", package)
else:
    print("\n✅ All required packages are available!")


PACKAGE CHECK
✅ PyMuPDF
✅ pytesseract
✅ faiss-cpu
✅ sentence-transformers
✅ langchain-groq
✅ langchain-text-splitters
✅ langgraph

✅ All required packages are available!


In [22]:
process = subprocess.run(
    [sys.executable, AGENT_FILE],
    capture_output=True,
    text=True,
    timeout=60
)

print("\n" + "=" * 60)
print("RETURN CODE:", process.returncode)
print("=" * 60)

if process.stdout:
    print("\nSTDOUT:\n")
    print(process.stdout)

if process.stderr:
    print("\nSTDERR:\n")
    print(process.stderr)


RETURN CODE: 1

STDERR:

  File "c:\Users\hp\OneDrive\Desktop\utsarjan\generated_agent.py", line 238
    answer_with_citations = f"{raw}
                            ^
SyntaxError: unterminated string literal (detected at line 238)

